# LegalCheck India — Notebook 1: Pipeline 1 (fine-tuned text classification)
**Group 10 · ISOM5240 · Gollakaram Vishnu Srivardhan (21401910, leader) & Rahul Singh (2610gnam02)**

**Objective of this pipeline:** decide whether a client's English narrative describes a **legally valid dispute / issue in India** (`VALID`) or not (`NOT_VALID`).

**Steps**
1. Load the frozen, balanced dataset (5,088 train / 994 validation / 1,008 test). Validation and test use *scenario families never seen in training*.
2. Fine-tune **three comparable pre-trained base models** with the same data, settings and seed:
   `distilbert/distilbert-base-uncased`, `google-bert/bert-base-uncased`, `law-ai/InLegalBERT`.
3. Record **accuracy** (`trainer.evaluate()` on validation, plus the unseen test set) and **runtime** (training and inference).
4. Select the best model (rule: highest validation accuracy; tie → fastest inference), save it, test reload, and upload it to Hugging Face.

Runtime: Colab **T4 GPU** (Runtime → Change runtime type → T4 GPU). Expected run time ≈ 10–15 minutes.

In [ ]:
# Step 0 - Install packages (versions compatible with the course notebooks: transformers < 5)
!pip uninstall -y -q wandb
!pip install -q "transformers>=4.51.0,<5.0.0" "accelerate>=0.34.0" scikit-learn pandas openpyxl

In [ ]:
# ---- Hardware and library versions (recorded for reproducibility) ----
import platform, torch, transformers
DEVICE = 0 if torch.cuda.is_available() else -1
print(f"Python {platform.python_version()} | torch {torch.__version__} | transformers {transformers.__version__}")
print(f"Device: {torch.cuda.get_device_name(0) if DEVICE == 0 else 'CPU'}")

In [ ]:
# ---- Get the project files (dataset CSVs) ----
# Works three ways, all using the SAME frozen CSV files:
#   (1) CSVs uploaded anywhere under /content (or the submitted data folder) are found automatically;
#   (2) an uploaded Colab_pack.zip / GitHub_repo_files.zip is unzipped automatically;
#   (3) otherwise the public GitHub repository (REPO_URL) is cloned.
import glob, os, subprocess, zipfile
REPO_URL = "https://github.com/vishnusrivardhangollakaram-boop/Group10_DLBAWP_Group_Project.git"
SEARCH_ROOTS = [os.getcwd(), "/content", ".."]


def find_data_folder():
    """Return the first folder (searched recursively) that contains train.csv, or None."""
    for root in SEARCH_ROOTS:
        if not os.path.isdir(root):
            continue
        for dirpath, dirnames, filenames in os.walk(root):
            dirnames[:] = [d for d in dirnames if d not in ("sample_data", "drive", ".config", "checkpoints")]
            if "train.csv" in filenames and "validation.csv" in filenames:
                return dirpath
    return None


DATA_FOLDER = find_data_folder()
if DATA_FOLDER is None:                                   # (2) unzip an uploaded pack if present
    for archive in glob.glob("/content/*.zip") + glob.glob("*.zip"):
        with zipfile.ZipFile(archive) as handle:
            handle.extractall("/content/unzipped_data")
    DATA_FOLDER = find_data_folder()
if DATA_FOLDER is None:                                   # (3) clone the public repository
    if "OWNER/REPO" in REPO_URL:
        raise FileNotFoundError("Upload train.csv, validation.csv, test.csv, app_test_cases.csv and "
                                "drafting_eval_cases.csv to the Colab Files panel, then run this cell again.")
    subprocess.run(["git", "clone", "-q", REPO_URL, "/content/repo"], check=True)
    DATA_FOLDER = find_data_folder()
print(f"Using data folder: {DATA_FOLDER}")
print(sorted(f for f in os.listdir(DATA_FOLDER) if f.endswith(".csv")))

In [ ]:
# Step 1 - Settings (all experiment settings in one place)
import json, random, time
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix

CANDIDATE_MODELS = [
    "distilbert/distilbert-base-uncased",   # compact general-English baseline (used in the course)
    "google-bert/bert-base-uncased",        # full-size general-English BERT
    "law-ai/InLegalBERT",                   # BERT pre-trained on Indian legal text
]
LABEL_NAMES = ["NOT_VALID", "VALID"]                     # label 0, label 1
ID2LABEL = {0: "NOT_VALID", 1: "VALID"}
LABEL2ID = {"NOT_VALID": 0, "VALID": 1}
MAX_LENGTH = 128          # narratives are short; same value is used in the app
NUM_EPOCHS = 3            # compare 1-3 epochs; the best epoch on validation is kept
BATCH_SIZE = 32
LEARNING_RATE = 3e-5
SEED = 5240
HF_MODEL_ID = "VishnuGollakaram/legalcheck-india-validity"   # destination on Hugging Face
RESULTS_FOLDER = "results"
os.makedirs(RESULTS_FOLDER, exist_ok=True)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

In [ ]:
# Step 2 - Load and inspect the dataset
train_df = pd.read_csv(os.path.join(DATA_FOLDER, "train.csv"))
val_df = pd.read_csv(os.path.join(DATA_FOLDER, "validation.csv"))
test_df = pd.read_csv(os.path.join(DATA_FOLDER, "test.csv"))
for split_name, frame in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    print(f"{split_name:10s} rows={len(frame):5d} | VALID={int(frame.label.sum()):5d} | NOT_VALID={int((frame.label == 0).sum()):5d} | families={frame.family.nunique()}")
print("\nFamily overlap train/val:", set(train_df.family) & set(val_df.family), "| train/test:", set(train_df.family) & set(test_df.family))
train_df.sample(5, random_state=SEED)[["text", "label_name", "family"]]

In [ ]:
# Step 3 - Helper functions: torch Dataset (as in the course notebook) and metrics
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer,
                          pipeline)

class NarrativeDataset(torch.utils.data.Dataset):
    """Course-style torch Dataset: tokenised encodings + integer labels."""
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)


def make_dataset(tokenizer, frame):
    """Tokenise one split with the fixed MAX_LENGTH."""
    encodings = tokenizer(list(frame.text), padding="max_length", truncation=True, max_length=MAX_LENGTH)
    return NarrativeDataset(encodings=encodings, labels=list(frame.label))


def compute_metrics(eval_pred):
    """Accuracy (main metric) plus F1 for the VALID class."""
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, predictions), "f1": f1_score(labels, predictions)}


def time_single_inference(classifier, texts):
    """Warm up once, then time one-by-one predictions (how the app is used). Returns mean seconds."""
    classifier(texts[0], truncation=True, max_length=MAX_LENGTH)            # warm-up, not timed
    durations = []
    for text in texts:
        start_time = time.perf_counter()
        classifier(text, truncation=True, max_length=MAX_LENGTH)
        durations.append(time.perf_counter() - start_time)
    return float(np.mean(durations))

In [ ]:
# Step 4 - Fine-tune and evaluate one candidate (same data, settings and seed for every model)
def fine_tune_candidate(model_name):
    """Fine-tune one base model; return (summary dict, per-epoch history DataFrame, trainer, tokenizer)."""
    print(f"\n========== {model_name} ==========")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2,
                                                               id2label=ID2LABEL, label2id=LABEL2ID)
    train_dataset = make_dataset(tokenizer=tokenizer, frame=train_df)
    val_dataset = make_dataset(tokenizer=tokenizer, frame=val_df)
    output_dir = f"checkpoints/{model_name.split('/')[-1]}"
    args = TrainingArguments(output_dir=output_dir, num_train_epochs=NUM_EPOCHS,
                             per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=64,
                             learning_rate=LEARNING_RATE, weight_decay=0.01, warmup_ratio=0.1,
                             eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
                             load_best_model_at_end=True, metric_for_best_model="accuracy",
                             logging_strategy="epoch", fp16=torch.cuda.is_available(),
                             report_to="none", seed=SEED)
    trainer = Trainer(model=model, args=args, train_dataset=train_dataset, eval_dataset=val_dataset,
                      compute_metrics=compute_metrics)
    train_result = trainer.train()
    val_metrics = trainer.evaluate()                         # best epoch reloaded (load_best_model_at_end)

    # Per-epoch history (train loss, validation loss, validation accuracy)
    history = pd.DataFrame(trainer.state.log_history)
    epoch_rows = []
    for epoch in sorted(history.epoch.dropna().unique()):
        rows = history[history.epoch == epoch]
        epoch_rows.append({"model": model_name, "epoch": int(round(epoch)),
                           "train_loss": rows["loss"].dropna().iloc[0] if "loss" in rows and rows["loss"].notna().any() else None,
                           "val_loss": rows["eval_loss"].dropna().iloc[0] if "eval_loss" in rows and rows["eval_loss"].notna().any() else None,
                           "val_accuracy": rows["eval_accuracy"].dropna().iloc[0] if "eval_accuracy" in rows and rows["eval_accuracy"].notna().any() else None})
    epoch_history = pd.DataFrame(epoch_rows).dropna(subset=["val_accuracy"])

    # Unseen-family test set: batch prediction with the trained model
    test_dataset = make_dataset(tokenizer=tokenizer, frame=test_df)
    start_time = time.perf_counter()
    test_output = trainer.predict(test_dataset)
    test_batch_seconds = time.perf_counter() - start_time
    test_predictions = np.argmax(test_output.predictions, axis=-1)

    # App-style single-text runtime (HF pipeline, 50 texts, after warm-up)
    classifier = pipeline("text-classification", model=trainer.model, tokenizer=tokenizer, device=DEVICE)
    mean_single_seconds = time_single_inference(classifier=classifier, texts=list(test_df.text[:50]))

    summary = {
        "model": model_name,
        "parameters_millions": round(sum(p.numel() for p in model.parameters()) / 1e6, 1),
        "best_epoch": int(round(epoch_history.loc[epoch_history.val_accuracy.idxmax(), "epoch"])),
        "val_accuracy": round(val_metrics["eval_accuracy"], 4),
        "val_f1": round(val_metrics["eval_f1"], 4),
        "test_accuracy": round(accuracy_score(test_df.label, test_predictions), 4),
        "test_f1": round(f1_score(test_df.label, test_predictions), 4),
        "test_precision": round(precision_score(test_df.label, test_predictions), 4),
        "test_recall": round(recall_score(test_df.label, test_predictions), 4),
        "train_runtime_sec": round(train_result.metrics["train_runtime"], 2),
        "test_batch_runtime_sec": round(test_batch_seconds, 2),
        "single_inference_sec": round(mean_single_seconds, 4),
    }
    print(json.dumps(summary, indent=2))
    test_detail = test_df.assign(model=model_name, predicted=test_predictions)
    return summary, epoch_history, trainer, tokenizer, test_detail

In [ ]:
# Step 5 - Run the three candidates
summaries, histories, test_details, trained = [], [], [], {}
for candidate in CANDIDATE_MODELS:
    try:
        summary, epoch_history, trainer, tokenizer, test_detail = fine_tune_candidate(model_name=candidate)
    except Exception as error:                     # a failed candidate is recorded, never hidden
        print(f"FAILED {candidate}: {error}")
        summaries.append({"model": candidate, "val_accuracy": float("nan"), "error": str(error)[:200]})
        continue
    summaries.append(summary); histories.append(epoch_history); test_details.append(test_detail)
    trained[candidate] = (trainer, tokenizer)

results_df = pd.DataFrame(summaries)
epochs_df = pd.concat(histories, ignore_index=True)
results_df.to_csv(f"{RESULTS_FOLDER}/pipeline1_model_comparison.csv", index=False)
epochs_df.to_csv(f"{RESULTS_FOLDER}/pipeline1_epoch_history.csv", index=False)
pd.concat(test_details).to_csv(f"{RESULTS_FOLDER}/pipeline1_test_predictions.csv", index=False)
results_df

In [ ]:
# Step 6 - Epoch history (1-3 epochs) for every candidate
epochs_df

In [ ]:
# Step 7 - Select the best model: highest validation accuracy; tie -> fastest single inference
ranked = results_df.dropna(subset=["val_accuracy"]).sort_values(by=["val_accuracy", "single_inference_sec"], ascending=[False, True]).reset_index(drop=True)
BEST_MODEL = ranked.loc[0, "model"]
print("Ranking:")
print(ranked[["model", "val_accuracy", "test_accuracy", "train_runtime_sec", "single_inference_sec"]].to_string(index=False))
print(f"\nSelected base model for fine-tuning: {BEST_MODEL}")

best_trainer, best_tokenizer = trained[BEST_MODEL]
best_predictions = pd.read_csv(f"{RESULTS_FOLDER}/pipeline1_test_predictions.csv").query("model == @BEST_MODEL")
print("Confusion matrix on unseen test families (rows = true NOT_VALID/VALID):")
print(confusion_matrix(best_predictions.label, best_predictions.predicted))

In [ ]:
# Step 8 - Save the selected fine-tuned model (this exact folder is uploaded to Hugging Face and used by the app)
FINAL_MODEL_FOLDER = "Fine-tuned_Model_files/legalcheck-india-validity"
best_trainer.save_model(FINAL_MODEL_FOLDER)
best_tokenizer.save_pretrained(FINAL_MODEL_FOLDER)
with open(f"{FINAL_MODEL_FOLDER}/training_summary.json", "w") as handle:
    json.dump({"base_model": BEST_MODEL, "max_length": MAX_LENGTH, "threshold": 0.5, "seed": SEED,
               "summary": ranked.loc[0].to_dict()}, handle, indent=2, default=str)
print(sorted(os.listdir(FINAL_MODEL_FOLDER)))

In [ ]:
# Step 9 - Reload the SAVED model in a fresh pipeline and test it (as the app will)
reloaded = pipeline("text-classification", model=FINAL_MODEL_FOLDER, device=DEVICE)
examples = ["Someone took Rs 20,000 from me promising a job and now refuses to return it.",
            "My friend did not like my new haircut and laughed at it."]
for text in examples:
    print(f"{reloaded(text, top_k=None)}  <-  {text}")

In [ ]:
# Step 10 - Upload to Hugging Face (needs a WRITE token saved as Colab secret "HF_TOKEN"; skipped otherwise)
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")
if HF_TOKEN:
    best_trainer.model.push_to_hub(HF_MODEL_ID, token=HF_TOKEN)
    best_tokenizer.push_to_hub(HF_MODEL_ID, token=HF_TOKEN)
    print(f"Uploaded: https://huggingface.co/{HF_MODEL_ID}")
else:
    print("HF_TOKEN not found - upload skipped (model saved locally in Fine-tuned_Model_files/).")

In [ ]:
# Step 11 - Package results and the model folder for the submission
!zip -q -r results_notebook1.zip results Fine-tuned_Model_files
print(open(f"{RESULTS_FOLDER}/pipeline1_model_comparison.csv").read())